# Memorization / Label Noise Benchmark

This notebook is a lightweight front end for the Python experiment pipeline in `experiments/`. The reusable code handles dataset loading, symmetric label noise, model training, activation extraction, MI/SMI estimation, aggregation, and plotting.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Generalization and Memorization":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebook_examples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

## Run a small clean vs noisy-label example

The full paper-matched defaults are SGD, learning rate `0.01`, momentum `0.9`, batch size `32`, `50` epochs, `m=500` SMI projections, and `n=10000` training samples. This notebook command uses smaller values so it is safe to try interactively.

In [ ]:
cmd = [
    sys.executable,
    "-m", "experiments.run_label_noise",
    "--setup", "memorization",
    "--dataset", "mnist",
    "--model", "cnn5_bn",
    "--estimator", "smi_ksg_cd",
    "--epochs", "1",
    "--seeds", "0",
    "--noise-ratio", "0.4",
    "--conditions", "clean,noisy",
    "--max-mi-samples", "256",
    "--n-projs", "8",
    "--n-jobs", "-1",
    "--output-dir", str(OUTPUT_DIR),
    "--no-checkpoints",
]
print(" ".join(cmd))
# subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## Load and inspect results

In [ ]:
results_path = OUTPUT_DIR / "label_noise" / "mnist" / "cnn5_bn" / "smi_ksg_cd" / "results.csv"
if results_path.exists():
    results = pd.read_csv(results_path)
    display(results.head())
else:
    print(f"Run the command above first. Missing: {results_path}")

## Aggregate and plot clean vs noisy estimates

In [ ]:
aggregate_path = OUTPUT_DIR / "label_noise_summary.csv"
aggregate_cmd = [sys.executable, "-m", "experiments.aggregate_results", "--input", str(results_path), "--output", str(aggregate_path)]
plot_cmd = [sys.executable, "-m", "experiments.plot_results", "--input", str(aggregate_path), "--output-dir", str(OUTPUT_DIR / "figures"), "--x", "epoch", "--hue", "condition"]
print(" ".join(aggregate_cmd))
print(" ".join(plot_cmd))
# subprocess.run(aggregate_cmd, cwd=PROJECT_ROOT, check=True)
# subprocess.run(plot_cmd, cwd=PROJECT_ROOT, check=True)

## Direct module example

In [ ]:
from experiments.datasets import load_dataset
from experiments.noise import apply_symmetric_label_noise
from experiments.utils import sample_indices, set_global_seed

set_global_seed(0)
data = load_dataset("mnist")
noisy_labels, corruption_mask = apply_symmetric_label_noise(data.y_train, data.num_classes, noise_ratio=0.4, seed=0)
idx = sample_indices(len(data.x_train), max_items=256, seed=0)
print(data.input_shape, data.num_classes)
print("observed corruption rate:", corruption_mask.mean())
print("training analysis subset:", idx.shape)